In [1]:
import os
import sys
import requests
import datetime as dt
import numpy as np
from dotenv import load_dotenv
from pathlib import Path
import requests
import pandas as pd
import talib as ta
import plotly.graph_objs as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Text, Button, Output
from IPython.display import display

from Modules.utility import Utility
from Modules.show_plot import ShowPlot
from Modules.reques_api import RequestApi
from Modules.get_market_data import GetMarketData
from Modules.stock_prices_and_market_data import ClassStockPricesMarketData
from Modules.financial import Financial
from Modules.pdf_url_to_markdown import PDFUrlToMarkdown
from Modules.webpage_to_markdown import WebpageToMarkdown

In [2]:
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
request_api = RequestApi(API_BASE_URL)
get_market_data = GetMarketData(Path('/workspace/data'))
utility = Utility()
stock_prices_market_data = ClassStockPricesMarketData()
financial = Financial()
# URLからPDFをダウンロードしてMarkdownに変換するクラスのインスタンスを作成
pdf_to_md = PDFUrlToMarkdown()
# WEBページをMarkdownに変換する関数
webpage_to_markdown = WebpageToMarkdown()

In [3]:
code = 'FUU'
market = 'V'
start = '1999-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')

response = request_api.update_stock_timeseries_data(
    code=code,
    market=market,
    start=start,
    end=end
)
response

{'result': True}

In [4]:
dsv_timeseries_df = request_api.get_stock_time_series_data(
    code=code,
    market=market,
    start=start,
    end=end
)
dsv_timeseries_df

取得件数: 3112


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,upper1,lower1,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1184063,FUU,V,2013-12-10,0.540,1.200,0.480,0.480,201100,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1184064,FUU,V,2013-12-11,0.620,0.740,0.460,0.600,353000,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1184065,FUU,V,2013-12-12,0.640,0.660,0.560,0.660,52350,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1184066,FUU,V,2013-12-13,0.640,0.680,0.620,0.660,45950,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1184067,FUU,V,2013-12-16,0.420,0.560,0.420,0.560,569750,0.592,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3107,1187170,FUU,V,2026-05-04,0.185,0.190,0.180,0.190,361000,0.185,...,0.197677,0.169923,True,NaN,NaN,NaN,NaN,NaN,NaN,False
3108,1187171,FUU,V,2026-05-05,0.185,0.190,0.178,0.190,982200,0.185,...,0.197808,0.172592,False,NaN,0.1852,NaN,NaN,NaN,NaN,False
3109,1187172,FUU,V,2026-05-06,0.185,0.190,0.180,0.185,643000,0.186,...,0.197402,0.175398,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3110,1187173,FUU,V,2026-05-07,0.180,0.190,0.175,0.185,1201300,0.187,...,0.197180,0.175220,True,0.187,NaN,NaN,NaN,NaN,NaN,False


In [5]:
def stock_prices_and_material_prices(
        code: str,
        name: str,
        start: str,
        end: str,
        df_sp: pd.DataFrame | None = None,
        df_mat1: pd.DataFrame | None = None,
        df_mat2: pd.DataFrame | None = None
    ):
    # /api/v1/time_series_data/stock/
    response = request_api.get_stock_time_series_data(
        code=code,
        market=None,
        start=start,
        end=end
    )
    df_stock = pd.DataFrame(response)

    # S&P500を統合
    if df_sp is not None:
        df_sp_tmp = df_sp.copy() if df_sp is not None else pd.DataFrame()
        if "date" not in df_sp_tmp.columns:
            df_sp_tmp = df_sp_tmp.reset_index()

        df_sp_tmp["date"] = pd.to_datetime(df_sp_tmp["date"])
        df_sp_tmp = df_sp_tmp.set_index("date")
        df_sp_tmp = df_sp_tmp.loc[start:end]

    # mat1価格を統合
    if df_mat1 is not None:
        df_mat1_tmp = df_mat1.copy() if df_mat1 is not None else pd.DataFrame()
        if "date" not in df_mat1_tmp.columns:
            df_mat1_tmp = df_mat1_tmp.reset_index()
        df_mat1_tmp["date"] = pd.to_datetime(df_mat1_tmp["date"])
        df_mat1_tmp = df_mat1_tmp.set_index("date")
        df_mat1_tmp = df_mat1_tmp.loc[start:end]

    # mat2価格を統合
    if df_mat2 is not None:
        df_mat2_tmp = df_mat2.copy() if df_mat2 is not None else pd.DataFrame()
        if "date" not in df_mat2_tmp.columns:
            df_mat2_tmp = df_mat2_tmp.reset_index()
        df_mat2_tmp["date"] = pd.to_datetime(df_mat2_tmp["date"])
        df_mat2_tmp = df_mat2_tmp.set_index("date")
        df_mat2_tmp = df_mat2_tmp.loc[start:end]

    # 金価格
    df = df_stock.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")
    df = df.loc[start:end]


    # インデックスを揃えて結合
    if df_sp is not None:
        df["MA5_SP"] = df_sp_tmp["ma5"].reindex(df.index)
        df["MA25_SP"] = df_sp_tmp["ma25"].reindex(df.index)
    if df_mat1 is not None:
        df["MA5_MAT1"] = df_mat1_tmp["ma5"].reindex(df.index)
        df["MA25_MAT1"] = df_mat1_tmp["ma25"].reindex(df.index)
    if df_mat2 is not None:
        df["MA5_MAT2"] = df_mat2_tmp["ma5"].reindex(df.index)
        df["MA25_MAT2"] = df_mat2_tmp["ma25"].reindex(df.index)

    show_plot = ShowPlot()
    fig = show_plot.create_basic_chart(
        df=df.reset_index(),
        code=code,
        name=name,
        start=start,
        end=end
    )
    # ★ 2つのY軸を定義（左：HYMC、右：SP500 & GOLD）
    fig.update_layout(
        yaxis=dict(
            title=f"{name} Price",
            side="left"
        ),
        yaxis2=dict(
            title="SP500 / GOLD",
            overlaying="y",
            side="right"
        )
    )

    # --- SP500（右軸） ---
    if df_sp is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_SP"],
                name="SP_MA5",
                line={"color": "blue", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_SP"],
                name="SP_MA25",
                line={"color": "gray", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- MAT1（右軸） ---
    if df_mat1 is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_MAT1"],
                name="MAT1_MA5",
                line={"color": "orange", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_MAT1"],
                name="MAT1_MA25",
                line={"color": "yellow", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- MAT2（左軸） ---
    if df_mat2 is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_MAT2"],
                name="MAT2_MA5",
                line={"color": "gray", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_MAT2"],
                name="MAT2_MA25",
                line={"color": "black", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )

    return fig

In [6]:
name = "Silver Mountain Resources Inc"
start = dt.datetime(2025, 1, 1).strftime("%Y-%m-%d")
end = dt.datetime(2026, 4, 17).strftime("%Y-%m-%d")
# グラフ領域の作成
fig = stock_prices_and_material_prices(
    code=code,
    name=name,
    start=start,
    end=end,
    df_sp=None,
    df_mat1=None,
    df_mat2=None
)
fig.show()

取得件数: 531


In [7]:
response = request_api.update_corp_finance_data(
    code=code,
    market=market
)
response

{'result': True}

In [8]:
fuu_financials_data = request_api.get_corp_financials_data(code=code, market=market)
fuu_balance_sheet_data = request_api.get_corp_balance_sheet_data(code=code, market=market)
fuu_cash_flow_data = request_api.get_corp_cash_flow_data(code=code, market=market)
fuu_earnings_data = request_api.get_corp_earnings_data(code=code, market=market)
fuu_quarterly_earnings_data = request_api.get_corp_quarterly_earnings_data(code=code, market=market)

In [9]:
# ４年分の財務データ
fuu_financials_data_df = pd.DataFrame(fuu_financials_data['results'])
# ４年分のバランスシート
fuu_balance_sheet_data_df = pd.DataFrame(fuu_balance_sheet_data['results'])
# ４年分のキャッシュフロー
fuu_cash_flow_data_df = pd.DataFrame(fuu_cash_flow_data['results'])
# ４年分の収益データ
fuu_earnings_data_df = pd.DataFrame(fuu_earnings_data['results'])
# ４年分の四半期収益データ
fuu_quarterly_earnings_data_df = pd.DataFrame(fuu_quarterly_earnings_data['results'])

In [10]:
"""
◆ 1. 株価・市場データ
• 現在株価（Price）
• 時価総額（Market Cap）
• 出来高（Volume）
• 52週高値・安値
• Beta（ボラティリティ指標）ß
"""
stock_prices_and_market_data = stock_prices_market_data.stock_prices_and_market_data(
    code=code,
    market=market,
    bs_df=fuu_balance_sheet_data_df
)
stock_prices_and_market_data.to_markdown()

取得件数: 459
取得件数: 460
取得件数: 458
取得件数: 459
取得件数: 458
取得件数: 458
取得件数: 457
取得件数: 456


'|    |   close |   market_cap |   shares_outstanding |   higher_rate_par_52_weeks |   lower_rate_par_52_weeks |     beta |\n|---:|--------:|-------------:|---------------------:|---------------------------:|--------------------------:|---------:|\n|  0 |   0.07  |  1.2764e+07  |          1.82343e+08 |                      0.165 |                     0.025 | 0.788011 |\n|  1 |   0.085 |  2.52047e+07 |          2.96526e+08 |                      0.3   |                     0.055 | 1.32672  |\n|  2 |   0.165 |  6.01138e+07 |          3.64326e+08 |                      0.51  |                     0.065 | 1.45504  |\n|  3 |   0.13  |  6.41358e+07 |          4.93353e+08 |                      0.54  |                     0.065 | 0.441975 |'

In [11]:
"""
◆ 2. 財務データ（Financials）+ EPS（Earnings Per Share）+ PBR（Price-to-Book Ratio）
• 売上高（Revenue）
• 営業利益（Operating Income）
• 純利益（Net Income）
• EBITDA（企業による）
• 総資産（Total Assets）
• 総負債（Total Liabilities）
• 現金（Cash）
• 希釈EPS（Diluted EPS）
• 基本EPS（Basic EPS）
• 営業キャッシュフロー（Operating Cash Flow）
• フリーキャッシュフロー（Free Cash Flow）
"""
financial_df = financial.calc_financial(
    code = code,
    market = market,
)
financial_df.to_markdown()

取得件数: 2042


'|    | date                |   revenue |          earnings |   total_assets |       total_debt |   cash_and_cash_equivalents |            EBITDA |   operating_income |   basic_eps |   diluted_eps |   operating_cash_flow |    free_cash_flow |\n|---:|:--------------------|----------:|------------------:|---------------:|-----------------:|----------------------------:|------------------:|-------------------:|------------:|--------------:|----------------------:|------------------:|\n|  0 | 2021-06-30 00:00:00 |         0 | -828642           |    1.34356e+07 |      0           |                 1.69495e+06 | -825549           |  -824653           |      nan    |        nan    |     -870258           | -979360           |\n|  1 | 2022-06-30 00:00:00 |         0 |      -5.67012e+06 |    2.99369e+07 | 122282           |                 1.26181e+07 |      -3.00319e+06 |       -5.12954e+06 |       -0.02 |         -0.02 |          -3.6001e+06  |      -1.01459e+07 |\n|  2 | 2023-06-30 00:00:00 

In [12]:
# PBR（Price-to-Book Ratio）やROE（Return on Equity）などの投資指標を計算
financial.calc_stock_investment_indicators(code=code, market=market).to_markdown()

取得件数: 457


'|    | date                |   EV | reason   |         ROE |   operating_income |   basic_eps |   diluted_eps |\n|---:|:--------------------|-----:|:---------|------------:|-------------------:|------------:|--------------:|\n|  0 | 2021-06-30 00:00:00 |      | no_price |  -0.0620134 |  -824653           |      nan    |        nan    |\n|  1 | 2022-06-30 00:00:00 |      | no_price |  -0.19983   |       -5.12954e+06 |       -0.02 |         -0.02 |\n|  2 | 2023-06-30 00:00:00 |      | no_price |  -0.229984  |       -1.19892e+07 |       -0.03 |         -0.03 |\n|  3 | 2024-06-30 00:00:00 |      | no_price |  -0.279161  |       -1.86533e+07 |       -0.05 |         -0.05 |\n|  4 | 2025-06-30 00:00:00 |  nan | nan      | nan         |      nan           |       -0.03 |         -0.03 |'

In [13]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://f3uranium.com/wp-content/uploads/2025/12/F3-MDA-Q1-2026-FINAL.pdf",
    directory_path="/workspace/data",
)
md_file_path

Generated: /workspace/data/F3-MDA-Q1-2026-FINAL.pdf.md


'/workspace/data/F3-MDA-Q1-2026-FINAL.pdf.md'

In [14]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://f3uranium.com/wp-content/uploads/2025/12/F3-FS-Q1-2026-FINAL.pdf",
    directory_path="/workspace/data",
)
md_file_path

Generated: /workspace/data/F3-FS-Q1-2026-FINAL.pdf.md


'/workspace/data/F3-FS-Q1-2026-FINAL.pdf.md'

In [ ]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="",
    directory_path="/workspace/data",
)
md_file_path

In [ ]:
web = webpage_to_markdown.webpage_to_markdown(
    url="",
    directory_path="/workspace/data",
    timeout=120
)
web

ReadTimeout: HTTPSConnectionPool(host='ir.eaglenuclear.com', port=443): Read timed out. (read timeout=120)